# Analyize Results Parsed from Log Files

### Experiments

#### Exploring Different Models

- googlenet baseline run

- mobilenet_v3_small baseline run

- restnet50 baseline run

- vgg16 baseline run

#### Data Augmentation 

- googlenet color aug

- googlenet color jitter

- googlenet random erase

- googlenet all augs

In [6]:
import pandas as pd 
import matplotlib.pyplot as plt
from pathlib import Path
import os

In [7]:
# Discover all training CSV files
results_dir = Path('results')
training_csvs = list(results_dir.glob('*_training.csv'))

print(f"Found {len(training_csvs)} training CSV files:")
for csv_file in training_csvs:
    print(f"  - {csv_file.name}")

Found 8 training CSV files:
  - googlenet-all-augs-veri_training.csv
  - googlenet-color-aug-veri_training.csv
  - googlenet-color-jitter-veri_training.csv
  - googlenet-random-erase-veri_training.csv
  - googlenet-veri_training.csv
  - mobilenet_v3_small-veri_training.csv
  - resnet50-veri_training.csv
  - vgg16-veri_training.csv


In [ ]:
def plot_epoch_accuracy(experiment_name, df, save_dir='figures'):
    """
    Plot final acc_avg for each epoch (one point per epoch).
    
    Args:
        experiment_name: Name of the experiment (for title and filename)
        df: DataFrame with columns 'epoch', 'acc_avg'
        save_dir: Directory to save the plot
    """
    # Get the last row of each epoch (final acc_avg for that epoch)
    epoch_final = df.groupby('epoch').last().reset_index()
    
    # Create figure
    fig, ax = plt.subplots(figsize=(12, 6))
    
    # Plot accuracy
    ax.plot(epoch_final['epoch'], epoch_final['acc_avg'], 
            linewidth=2, color='#2E86AB', marker='o', markersize=6)
    
    # Formatting
    ax.set_xlabel('Epoch', fontsize=12)
    ax.set_ylabel('Average Accuracy (%)', fontsize=12)
    ax.set_title(f'{experiment_name} - Final Accuracy per Epoch', fontsize=14, fontweight='bold')
    ax.grid(True, alpha=0.3)
    ax.set_xticks(epoch_final['epoch'])
    
    plt.tight_layout()
    
    # Save figure
    os.makedirs(save_dir, exist_ok=True)
    save_path = os.path.join(save_dir, f'{experiment_name}_epoch_accuracy.png')
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    print(f"Saved: {save_path}")
    
    # Display
    plt.show()


def plot_batch_instant_accuracy(experiment_name, df, save_dir='figures'):
    """
    Plot acc_instant for all batches (detailed batch-level view).
    
    Args:
        experiment_name: Name of the experiment (for title and filename)
        df: DataFrame with columns 'epoch', 'batch', 'total_batches', 'acc_instant'
        save_dir: Directory to save the plot
    """
    # Calculate continuous epoch position
    df['epoch_position'] = df['epoch'] + (df['batch'] / df['total_batches'])
    
    # Create figure
    fig, ax = plt.subplots(figsize=(12, 6))
    
    # Plot instant accuracy
    ax.plot(df['epoch_position'], df['acc_instant'], 
            linewidth=1, color='#A23B72', alpha=0.7)
    
    # Formatting
    ax.set_xlabel('Epoch', fontsize=12)
    ax.set_ylabel('Instant Accuracy (%)', fontsize=12)
    ax.set_title(f'{experiment_name} - Batch-level Instant Accuracy', fontsize=14, fontweight='bold')
    ax.grid(True, alpha=0.3)
    
    # Set x-axis to show integer epoch numbers
    max_epoch = int(df['epoch'].max())
    ax.set_xticks(range(1, max_epoch + 1))
    
    plt.tight_layout()
    
    # Save figure
    os.makedirs(save_dir, exist_ok=True)
    save_path = os.path.join(save_dir, f'{experiment_name}_batch_instant_accuracy.png')
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    print(f"Saved: {save_path}")
    
    # Display
    plt.show()

In [ ]:
# Generate both plots for all experiments
for csv_file in training_csvs:
    # Extract experiment name from filename
    experiment_name = csv_file.stem.replace('_training', '')
    
    # Load CSV
    df = pd.read_csv(csv_file)
    
    print(f"\n{'='*60}")
    print(f"Processing: {experiment_name}")
    print(f"{'='*60}")
    
    # Plot 1: Final accuracy per epoch (clean view)
    print("\n1. Epoch-level accuracy (final acc_avg per epoch):")
    plot_epoch_accuracy(experiment_name, df)
    
    # Plot 2: Batch-level instant accuracy (detailed view)
    print("\n2. Batch-level instant accuracy:")
    plot_batch_instant_accuracy(experiment_name, df)

print(f"\n{'='*60}")
print(f"✓ Generated {len(training_csvs) * 2} plots in figures/ directory")
print(f"  - {len(training_csvs)} epoch-level plots")
print(f"  - {len(training_csvs)} batch-level plots")
print(f"{'='*60}")